# NB14 – Produktsteckbrief: Heimspeicher 7 kW / 4 kWh
### CAS Information Engineering – Scripting Project (Kür)
**Gruppe:** SC26_Gruppe_2 | **Datum:** März–Mai 2026

---
Wirtschaftlichkeitsanalyse eines **konkreten Produkts** mit folgenden Spezifikationen:

| Parameter | Wert |
|---|---|
| **Konverterleistung** | 7 kW |
| **Kapazität** | 4 kWh |
| **Verkaufspreis** | **2 000 EUR** (~2 060 CHF) |
| **CAPEX/kWh** | ~500 EUR/kWh |

Vergleich gegen die vier Projektsegmente aus NB04; Simulation für Grid-Arbitrage
und Eigenverbrauchsoptimierung (NB13).

---
| [↑ Projektübersicht](00_Project_Overview.ipynb) | [← NB13 Eigenverbrauch](13_Eigenverbrauch.ipynb) |
|:---|---:|


---
## 0. Setup

Bibliotheken laden, `config.json` lesen, Produktparameter
(7 kW / 4 kWh, LFP, RTE 92%) und Transfer-Daten aus NB13 einlesen.


In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import os, json as _json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

with open('config.json') as _f:
    CFG = _json.load(_f)

SZ_AKTIV   = CFG['szenarien']['gleichzeitigkeit_aktiv']
FORCE_RELOAD = CFG.get('force_reload', {})  # konventionskonform gelesen
DIR_INTER  = os.path.join('data', 'intermediate', SZ_AKTIV)
CHARTS_DIR = os.path.join('output', 'charts', SZ_AKTIV)
os.makedirs(CHARTS_DIR, exist_ok=True)
DPI = CFG['visualisierung']['output_dpi']  # SSOT: config.json

_w         = CFG['pflicht']['wirtschaftlichkeit']
OPEX_RATE  = _w['opex_rate']
LIFETIME   = _w['lifetime_j']
ZIEL_ROI   = round(100 / LIFETIME, 2)    # = 1/lifetime_j: ROI für BE in LIFETIME Jahren

# ── Farben & Stil aus config.json (SSOT) ─────────────────────────────────────
# Bestehende Variablen (Rückwärtskompatibilität)
_viz        = CFG.get('visualisierung', {}).get('farben', {})
BG_DARK     = _viz.get('bg_dark',    '#0d1117')
BG_PANEL    = _viz.get('bg_panel',   '#141414')
C_PRICE     = _viz.get('c_price',    '#FFA726')
C_LOAD      = _viz.get('c_load',     '#66BB6A')
C_CHARGE    = _viz.get('c_charge',   '#1565C0')
C_FEED      = _viz.get('c_feed',     '#B71C1C')
SEG_COLORS  = _viz.get('seg_colors', ['#42A5F5', '#66BB6A', '#FFA726', '#EF5350'])
C_PRIV, C_GEW, C_IND, C_UTIL = SEG_COLORS

# UI-Strukturfarben
C_ACHSE      = _viz.get('c_achse',      '#aaaaaa')  # Achsenbeschriftungen
C_TICK       = _viz.get('c_tick',       '#bbbbbb')  # Tick-Labels
C_SPINE      = _viz.get('c_spine',      '#333333')  # Achsenrahmen
C_LEGENDE_BG = _viz.get('c_legende_bg', '#111111')  # Legenden-Hintergrund
C_GITTER     = _viz.get('c_gitter',     '#cccccc')  # Gitterlinien

# Funktionale Extrafarben (nur laden was das NB braucht)
C_DISPATCH   = _viz.get('c_dispatch',   '#AB47BC')  # Dispatch-optimal
C_STACKING   = _viz.get('c_stacking',   '#5DCAA5')  # Revenue Stacking
C_SOLAR      = _viz.get('c_solar',      '#FDD835')  # Solar-Ertrag
C_GRENZWERT  = _viz.get('c_amber_dark', '#FF6F00')  # Grenzwert / Warnung
C_CYAN       = _viz.get('c_cyan',       '#26C6DA')  # Flusswasser / Alt. Speicher
C_GRUEN_DARK = _viz.get('c_gruen_dark', '#388E3C')  # Erneuerbare

# Stilkonstanten
_stil               = CFG.get('visualisierung', {}).get('stil', {})
LW                  = _stil.get('linienbreite_standard', 1.5)   # Standard-Linienbreite
LW_DUENN            = _stil.get('linienbreite_duenn',    0.8)   # dünne Linien
LW_DICK             = _stil.get('linienbreite_dick',     2.5)   # dicke Linien
ALPHA_FLAECHE       = _stil.get('alpha_flaeche',         0.12)  # dezente Füllung
ALPHA_FLAECHE_STARK = _stil.get('alpha_flaeche_stark',   0.35)  # Balken / Füllung
ALPHA_LEGENDE       = _stil.get('alpha_legende',         0.30)  # Legenden-BG
ALPHA_GEDAEMPFT     = _stil.get('alpha_linie_gedaempft', 0.55)  # Nebenlinien
FS_TITEL            = _stil.get('schriftgroesse_titel',   13)   # Chart-Titel
FS_ACHSE            = _stil.get('schriftgroesse_achse',   10)   # Achsenbeschr.
FS_TICK             = _stil.get('schriftgroesse_tick',     9)   # Ticks
FS_LEGENDE          = _stil.get('schriftgroesse_legende',  8)   # Legende
FS_KLEIN            = _stil.get('schriftgroesse_klein',    7)   # Annotationen

# matplotlib rcParams — nur stabile, versionsunabhängige Keys (matplotlib >= 3.5)
# axes.titlecolor (3.8+) und axes.grid (stört Karten) bewusst NICHT gesetzt
import matplotlib as _mpl
_mpl.rcParams.update({
    'figure.facecolor':  BG_DARK,
    'axes.facecolor':    BG_PANEL,
    'axes.edgecolor':    C_SPINE,
    'axes.labelcolor':   C_ACHSE,
    'axes.labelsize':    FS_ACHSE,
    'axes.titlesize':    FS_TITEL,
    'xtick.color':       C_TICK,
    'ytick.color':       C_TICK,
    'xtick.labelsize':   FS_TICK,
    'ytick.labelsize':   FS_TICK,
    'text.color':        'white',
    'lines.linewidth':   LW,
    'legend.facecolor':  C_LEGENDE_BG,
    'legend.framealpha': ALPHA_LEGENDE,
    'legend.fontsize':   FS_LEGENDE,
    'legend.edgecolor':  C_SPINE,
})
print('Farben & Stil geladen.')

# -- Transfer: Ergebnisse aus transfer.json laden ----------------------------
_tf_path = 'transfer.json'
if os.path.exists(_tf_path) and os.path.getsize(_tf_path) > 0:
    TF      = _json.load(open(_tf_path))
    TF_ECON = TF.get('simulation', {}).get('wirtschaftlichkeit', {})
    TF_HYB  = TF.get('hybrid_simulation', {}).get('ergebnisse', {})
    print(f"transfer.json geladen | Segmente: {list(TF_ECON.keys())}")
else:
    TF = {}; TF_ECON = {}; TF_HYB = {}
    print('⚠️  transfer.json nicht gefunden — NB01/NB02 zuerst ausführen')

print(f"Setup OK | Szenario={SZ_AKTIV} | Charts → {CHARTS_DIR}")

**Produktparameter:** Technische Spezifikation des Referenzprodukts
(7 kW Leistung, 4 kWh Kapazität, LFP, RTE 92%) als Berechnungsbasis setzen.


In [ ]:
# ── Produktparameter ──────────────────────────────────────────────────────────
# Alle Werte in EUR (Projektwährung — Designentscheidung NB00 Sektion 4.4)
EUR_CHF      = CFG.get('eur_chf', 0.97)   # Fixkurs aus config.json (SSOT)
PROD_NAME    = 'Heimspeicher (7 kW / 14 kWh)'
PROD_KW      = 7.0         # Konverterleistung [kW]
PROD_KWH     = 4.0        # Kapazität [kWh]
PROD_EUR     = 2000.0      # Verkaufspreis [EUR]

# Betriebsparameter — aus config.json (SSOT)
_sim14       = CFG['pflicht']['simulation']
EFFICIENCY   = _sim14['efficiency_roundtrip']
MIN_SOC      = _sim14['soc_min_pct']
MAX_SOC      = _sim14['soc_max_pct']
OPEX_RATE    = CFG['pflicht']['wirtschaftlichkeit']['opex_rate']
# LIFETIME und ZIEL_ROI: aus nb14_setup

# Abgeleitete Grössen
USABLE_KWH    = PROD_KWH * (MAX_SOC - MIN_SOC)
CAPEX_PER_KWH = PROD_EUR / PROD_KWH

print(f"Produkt:        {PROD_NAME}")
print(f"Preis:          {PROD_EUR:.0f} EUR  (~{PROD_EUR / EUR_CHF:.0f} CHF @ {EUR_CHF} EUR/CHF)")
print(f"CAPEX/kWh:      {CAPEX_PER_KWH:.0f} EUR/kWh")
print(f"Nutzbare Kap:   {USABLE_KWH:.1f} kWh  ({MIN_SOC*100:.0f}–{MAX_SOC*100:.0f} % SoC)")
print(f"C-Rate:         {PROD_KW/PROD_KWH:.2f}C  (Vollladung in {PROD_KWH/PROD_KW:.1f}h)")
# -- Transfer: Ergebnisse aus transfer.json laden ----------------------------
_tf_path = 'transfer.json'
if os.path.exists(_tf_path) and os.path.getsize(_tf_path) > 0:
    TF      = _json.load(open(_tf_path))
    _dt     = TF.get('datenzeitraum', {})
    _sim    = TF.get('simulation', {})
    TF_N_YEARS  = _dt.get('n_years', None)
    TF_START    = _dt.get('start_date', 'unbekannt')
    TF_END      = _dt.get('end_date', 'unbekannt')
    TF_SPREAD   = _sim.get('spread_mean_eur_mwh', None)
    TF_ECON     = _sim.get('wirtschaftlichkeit', {})
    TF_HYB      = TF.get('hybrid_simulation', {}).get('ergebnisse', {})
    TF_KUER     = CFG.get('kuer_aktiv', {})   # aus config.json (SSOT)
    TF_DISP     = TF.get('dispatch_optimierung', {})   # T4: NB10-Dispatch-ROI
    print(f"transfer.json: {TF_START} – {TF_END} ({TF_N_YEARS} Jahre) | Spread: {TF_SPREAD} EUR/MWh")
else:
    TF = {}; TF_N_YEARS = None; TF_START = TF_END = 'unbekannt'
    TF_SPREAD = None; TF_ECON = {}; TF_HYB = {}; TF_KUER = {}
    print('⚠️  transfer.json nicht gefunden — NB01/NB02 zuerst ausführen')

---
## 1. Einordnung in die Projektsegmente

Das Produkt liegt zwischen dem Privat-Segment (10 kWh / 5 kW) und dem Gewerbe-Segment
(100 kWh / 30 kW) — näher am Privat-Segment durch Kapazität, aber mit erhöhter
Leistung (7 kW statt 5 kW).


In [ ]:
# ── Einordnung: Produkt vs. Projektsegmente ───────────────────────────────────
seg_data = {
    'Segment'        : ['Privat 10 kWh', '>>> Dieses Produkt <<<', 'Gewerbe 100 kWh', 'Industrie 1 MWh'],
    'Kapazität [kWh]': [10,              PROD_KWH,                 100,               1000],
    'Leistung [kW]'  : [5,               PROD_KW,                  30,                200],
    'CAPEX [EUR]'    : [4000,            PROD_EUR,                  30000,             250000],
    'EUR/kWh'        : [400,             round(CAPEX_PER_KWH),      300,               250],
    'C-Rate'         : [0.5,             round(PROD_KW/PROD_KWH,2), 0.3,               0.2],
}
df_seg = pd.DataFrame(seg_data).set_index('Segment')
display(df_seg)
print(f"\n→ CAPEX/kWh dieses Produkts: {CAPEX_PER_KWH:.0f} EUR/kWh")
print(f"  Vergleich Privat-Segment:  400 EUR/kWh")
print(f"  → Produkt ist {(400-CAPEX_PER_KWH)/400*100:.0f}% günstiger pro kWh als Referenz-Privat")


---
## 2. Grid-Arbitrage: Wirtschaftlichkeit

Simulation mit denselben Parametern wie NB04 (p25/p75 Dispatch, alle Monate 2023/2024).
Da keine separate Simulation für dieses spezifische Produkt vorliegt, skalieren wir
die Ergebnisse linear aus dem Privat-Segment (10 kWh).


In [ ]:
# ── Grid-Arbitrage Kennzahlen ─────────────────────────────────────────────────
# Skalierung aus NB04-Ergebnissen (Privat 10 kWh als Basis)
# ROI und Erlös/kWh sind kapazitätsunabhängig → gleicher ROI, skalierter Absoluterlös

ROI_PRIVAT_PCT = 0.6    # % ROI laut NB04 Fazit (Privat, 2023/2024)

# Angepasster Erlös/Jahr für dieses Produkt
# C-Rate 1.0C statt 0.5C → mehr mögliche Zyklen pro Tag → leicht besserer Erlös
# Korrekturfaktor für höhere C-Rate (grob ~+20% mehr Zyklen nutzbar)
c_rate_korr = min(1.2, PROD_KW / PROD_KWH / 0.5)  # relativ zu Privat (0.5C)
net_annual_arb  = PROD_EUR * (ROI_PRIVAT_PCT/100) * c_rate_korr
opex_annual     = PROD_EUR * OPEX_RATE
netto_arb       = net_annual_arb - opex_annual
roi_arb         = netto_arb / PROD_EUR * 100
payback_arb     = PROD_EUR / netto_arb if netto_arb > 0 else float('inf')

print(f"── Grid-Arbitrage (Skalierung aus NB04, Privat) ──")
print(f"Basis-ROI Privat:          {ROI_PRIVAT_PCT:.1f} %/Jahr")
print(f"C-Rate-Korrekturfaktor:    {c_rate_korr:.2f}×")
print(f"Brutto Jahreserlös Arb.:   {net_annual_arb:.0f} EUR/Jahr")
print(f"OPEX (1.5% CAPEX):         {opex_annual:.0f} EUR/Jahr")
print(f"Netto Jahreserlös:         {netto_arb:.0f} EUR/Jahr")
print(f"ROI:                       {roi_arb:.1f} %/Jahr")
print(f"Break-Even:                {'> 20 J' if payback_arb > 20 else f'{payback_arb:.0f} J'}")


---
## 3. Eigenverbrauchsoptimierung: Wirtschaftlichkeit

Szenario: Haushalt mit HT/NT-Tarif, Tageslast ~12 kWh/Tag.
Die 14 kWh-Kapazität reicht für einen vollen Tag Eigenverbrauch — das ist ein Vorteil
gegenüber dem 10 kWh-Standard-Segment.


In [ ]:
# ── Eigenverbrauchsoptimierung ────────────────────────────────────────────────
# CH-Haushaltstarife: CHF-Werte × EUR_CHF aus config.json
HT_PREIS  = 0.30 * EUR_CHF  # EUR/kWh  (0.30 CHF/kWh)
NT_PREIS  = 0.22 * EUR_CHF  # EUR/kWh  (0.22 CHF/kWh)
DIFF_EUR  = HT_PREIS - NT_PREIS

VERBRAUCH_TAG_KWH = 12.0

lade_tag_kwh = min(USABLE_KWH, VERBRAUCH_TAG_KWH) / EFFICIENCY
eig_tag_kwh  = min(USABLE_KWH * EFFICIENCY, VERBRAUCH_TAG_KWH)

ersparnis_tag = eig_tag_kwh * DIFF_EUR
ersparnis_j   = ersparnis_tag * 365

opex_annual   = PROD_EUR * OPEX_RATE
netto_eig_j   = ersparnis_j - opex_annual
roi_eig       = netto_eig_j / PROD_EUR * 100
payback_eig   = PROD_EUR / netto_eig_j if netto_eig_j > 0 else float('inf')

print(f"── Eigenverbrauchsoptimierung ──")
print(f"Nutzbare Kapazität:        {USABLE_KWH:.1f} kWh")
print(f"Eigenverbrauch/Tag:        {eig_tag_kwh:.1f} kWh")
print(f"Tarif-Differenz:           {DIFF_EUR:.3f} EUR/kWh")
print(f"Ersparnis/Tag:             {ersparnis_tag:.2f} EUR")
print(f"Jahresersparnis:           {ersparnis_j:.0f} EUR/Jahr")
print(f"OPEX:                      {opex_annual:.0f} EUR/Jahr")
print(f"Netto-Jahresersparnis:     {netto_eig_j:.0f} EUR/Jahr")
print(f"ROI:                       {roi_eig:.1f} %/Jahr")
print(f"Break-Even:                {'> 20 J' if payback_eig > 20 else f'{payback_eig:.1f} J'}")


---
## 4. Kombinierter Business Case: Arbitrage + Eigenverbrauch

In der Praxis ist eine Kombination möglich: tagsüber Eigenverbrauch aus
dem NT-Netz, nachts verbleibende Kapazität für Arbitrage nutzen.


In [ ]:
# ── Kombinierter Case ─────────────────────────────────────────────────────────
# Vereinfacht: 70% Kapazität für Eigenverbrauch, 30% für Arbitrage
eig_anteil = 0.70
arb_anteil = 0.30

netto_kombi = (netto_eig_j * eig_anteil + netto_arb * arb_anteil)
roi_kombi   = netto_kombi / PROD_EUR * 100
payback_kombi = PROD_EUR / netto_kombi if netto_kombi > 0 else float('inf')

print(f"── Kombinierter Case (70 % Eigenverbrauch + 30 % Arbitrage) ──")
print(f"Netto Jahreserlös:         {netto_kombi:.0f} EUR/Jahr")
print(f"ROI:                       {roi_kombi:.1f} %/Jahr")
print(f"Break-Even:                {payback_kombi:.1f} Jahre")

# ── Übersichts-Chart ──────────────────────────────────────────────────────────
cases    = ['Grid-Arbitrage', 'Eigenverbrauch', 'Kombiniert']
roi_vals = [roi_arb, roi_eig, roi_kombi]
pb_vals  = [min(payback_arb, LIFETIME*2), min(payback_eig, LIFETIME*2), min(payback_kombi, LIFETIME*2)]
colors_c = [C_PRICE, C_LOAD, SEG_COLORS[0]]  # Arbitrage, EV, Kombiniert

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.patch.set_facecolor(BG_DARK)
for ax in (ax1, ax2):
    ax.set_facecolor(BG_PANEL); ax.tick_params(colors=C_TICK)
    for sp in ax.spines.values(): sp.set_edgecolor(C_SPINE)

# ROI
bars = ax1.bar(cases, roi_vals, color=colors_c, alpha=0.85)
ax1.axhline(ZIEL_ROI, color=C_UTIL, lw=LW, ls='--',
            label=f'BE-Ziel {LIFETIME}J ({ZIEL_ROI}%/J)')
for b, v in zip(bars, roi_vals):
    ax1.text(b.get_x()+b.get_width()/2, v+0.1, f'{v:.1f} %', ha='center', color='white', fontsize=11)
ax1.set_ylabel('ROI [%/Jahr]', color=C_ACHSE)
ax1.set_title(f'{PROD_NAME}\nROI nach Nutzungsmodell', color='white', fontweight='bold')
ax1.legend(fontsize=FS_TICK, framealpha=ALPHA_LEGENDE, facecolor=C_LEGENDE_BG, labelcolor='white')
ax1.grid(True, axis='y', alpha=0.15)

# Break-Even
bars2 = ax2.bar(cases, pb_vals, color=colors_c, alpha=0.85)
for b, v, orig in zip(bars2, pb_vals, [payback_arb, payback_eig, payback_kombi]):
    label = f'>{v:.0f} J' if orig > LIFETIME*2 else f'{v:.1f} J'
    ax2.text(b.get_x()+b.get_width()/2, v+0.3, label, ha='center', color='white', fontsize=11)
ax2.set_ylabel('Break-Even [Jahre]', color=C_ACHSE)
ax2.set_title(f'Break-Even nach Nutzungsmodell\n(Deckel: {LIFETIME*2} Jahre)', color='white', fontweight='bold')
ax2.axhline(LIFETIME, color=C_UTIL, lw=1.2, ls='--',
            label=f'BE-Ziel {LIFETIME}J')
ax2.legend(fontsize=FS_TICK, framealpha=ALPHA_LEGENDE, facecolor=C_LEGENDE_BG, labelcolor='white')
ax2.set_ylim(0, LIFETIME * 2 + 4)
ax2.grid(True, axis='y', alpha=0.15)

plt.tight_layout()
p = os.path.join(CHARTS_DIR, 'kuer_nb14_kennzahlen.png')
plt.savefig(p, dpi=DPI, bbox_inches='tight', facecolor=BG_DARK)
plt.show(); plt.close()
print(f"Gespeichert: {p}")

---
## 5. Zusammenfassung Produktsteckbrief

Automatisch generierter Steckbrief: alle Kennzahlen (Arbitrage, Eigenverbrauch,
Kombi) tabellarisch zusammengeführt; Ergebnis in `transfer.json` schreiben.


In [ ]:
# ── Steckbrief-Ausgabe ────────────────────────────────────────────────────────
print("=" * 55)
print(f"PRODUKTSTECKBRIEF — {PROD_NAME}")
print("=" * 55)
print(f"  Verkaufspreis:          {PROD_EUR:.0f} EUR  (~{PROD_EUR / EUR_CHF:.0f} CHF)")
print(f"  Kapazität:              {PROD_KWH:.0f} kWh")
print(f"  Konverterleistung:      {PROD_KW:.0f} kW")
print(f"  CAPEX/kWh:              {CAPEX_PER_KWH:.0f} EUR/kWh  ({'günstig' if CAPEX_PER_KWH < 250 else 'mittel'})")
print(f"  C-Rate:                 {PROD_KW/PROD_KWH:.2f}C")
print(f"  Nutzbare Kapazität:     {USABLE_KWH:.1f} kWh (SoC 5–95 %)")
print()
print("  WIRTSCHAFTLICHKEIT (CH 2023/2024):")
print(f"  Grid-Arbitrage:         ROI {roi_arb:.1f} %/J  |  Break-Even {'> 20 J' if payback_arb>20 else f'{payback_arb:.0f} J'}")
print(f"  Eigenverbrauchsopt.:    ROI {roi_eig:.1f} %/J  |  Break-Even {payback_eig:.1f} J")
print(f"  Kombiniert (70/30):     ROI {roi_kombi:.1f} %/J  |  Break-Even {payback_kombi:.1f} J")
print()
print("  EINORDNUNG:")
if roi_eig >= 5:
    print("  ✅  Eigenverbrauchsoptimierung erreicht Ziel-ROI 5 %")
else:
    print("  ⚠️  Keines der Modelle erreicht Ziel-ROI 5 % (2023/2024 Preise)")
if CAPEX_PER_KWH < 200:
    print("  ✅  CAPEX/kWh unter 200 EUR — sehr günstig für LFP-Heimspeicher")
elif CAPEX_PER_KWH < 300:
    print("  ✅  CAPEX/kWh unter 300 EUR — wettbewerbsfähig")
else:
    print("  ℹ️   CAPEX/kWh > 300 EUR — marktüblich, aber Potential für weiteren Rückgang")
print("=" * 55)


---

In [ ]:
# -- Transfer: Produktsteckbrief Ergebnisse -----------------------------------
import os as _os, json as _json  # Re-Import mit lokalem Alias (Transfer-Zelle)
_tf_path = 'transfer.json'
_tf = _json.loads(open(_tf_path).read() or '{}') if _os.path.exists(_tf_path) else {}
_tf.setdefault('produkt', {})
try:
    _tf['produkt'].update({
        'prod_name': PROD_NAME,
        'prod_eur': PROD_EUR,
        'capex_per_kwh': round(float(CAPEX_PER_KWH), 1),
        'roi_arb_pct': round(float(roi_arb), 2),
        'roi_ev_pct': round(float(roi_eig), 2),
        'roi_kombi_pct': round(float(roi_kombi), 2),
        'payback_kombi_j': round(float(payback_kombi), 1) if payback_kombi < 999 else 999,
    })
    print(f"transfer.json: produkt | {PROD_NAME} | Kombi-ROI: {roi_kombi:.1f}%")
except Exception as _e:
    print(f"transfer.json: produkt nicht geschrieben ({_e})")
with open(_tf_path, 'w') as _f: _json.dump(_tf, _f, indent=2, ensure_ascii=False)


In [ ]:
# os bereits in Setup geladen
# ── Abschlusskontrolle ───────────────────────────────────────────────────────
charts_out = [f for f in os.listdir(CHARTS_DIR) if f.startswith('kuer_nb14_')]
print("=== Abschlusskontrolle NB14 (Kür) ===")
for f in sorted(charts_out):
    print(f"  ✅  {f}")
print(f"  {len(charts_out)} Chart(s) erzeugt")
print("=== Ende ===")


---
| [↑ Projektübersicht](00_Project_Overview.ipynb) | [← NB13 Eigenverbrauch](13_Eigenverbrauch.ipynb) |
|:---|---:|
